# 从零手写 Transformer

目标：不调用 `nn.MultiheadAttention`，从零手写一个能训练的小型 Transformer，在 `0921/raw/cmn.txt` 中英翻译上跑通，最后和 0921 的 Seq2Seq + 注意力（BLEU-4 0.1389）做对照。

## 路线图（自底向上，完成一项勾一项）

- [ ] M1 两种 mask：padding mask、因果 mask
- [ ] M2 正弦位置编码
- [ ] M3 Scaled Dot-Product Attention（缩放 + mask）
- [ ] M4 Multi-Head Attention
- [ ] M5 Position-wise FFN
- [ ] M6 EncoderLayer / DecoderLayer（残差 + LayerNorm）
- [ ] M7 拼装 Encoder / Decoder / Transformer
- [ ] M8 整体自测：因果性检查 + 过拟合一个小 batch
- [ ] M9 接数据（复用 0921 词表与 DataLoader）
- [ ] M10 训练 + 推理 + BLEU 对照

## 约定（先定契约，再写实现）

- 记号：`b` batch、`l` 序列长、`d_model` 隐藏维、`h` 头数、`d_k = d_v = d_model / h`
- 形状契约
  - 单头注意力层：q `(b, h, lq, d_k)`、k `(b, h, lk, d_k)`、v `(b, h, lk, d_v)`；输出 `(b, h, lq, d_v)`
  - 多头层 / 两种 Layer / 整个模型：输入输出均为 `(b, l, d_model)`
  - 最终 logits：`(b, l, vocab)`
- mask 语义统一为：**True = 可见，False = 屏蔽**（实现时把屏蔽位填一个极大负数，softmax 后被遮处自然趋近 0）
- 每写完一个模块，先在随机张量上把自测跑绿，再写下一个
- 卡住时：先自己推一遍，再看同目录 `attention-is-all-you-need-pytorch/` 里对应文件的写法

## M0 · 环境与配置

先把「小配置」定下来，流程通了再加大：

- `d_model` 与词嵌入维度一致（残差连接要求同维）；`n_head = 4`，于是 `d_k = d_v = 32`
- `d_ff` 先取 2 倍 `d_model`（论文是 4 倍），层数 2（论文 6 层），为了显存和速度
- 特殊符沿用 0921 的 `Vocabulary`：PAD=0、SOS=1、EOS=2、UNK=3
- 下面这一格跑完，把 `d_k` 的值记熟：后面拆头 / 拼头全靠它

In [2]:
import math
import sys
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(10)

WEEK19 = Path.cwd()          # 打印出来若不是 week19 目录，手改成 r"d:\project\step1\week19"
assert (WEEK19 / "0921" / "raw" / "cmn.txt").exists(), f"路径不对: {WEEK19}"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class Cfg:
    # ── 模型 ──
    d_model = 128             # 与 0921 的 EMBEDDING_DIM 对齐，方便对照
    n_head = 4                # 先 4 头
    d_k = d_model // n_head   # 32
    d_v = d_model // n_head   # 32
    d_ff = 256                # 论文是 4 * d_model（512），先减半换速度
    n_layers = 2              # 论文 6 层
    dropout = 0.1
    max_len = 100             # 与数据预处理的最大长度一致（位置编码表按它建）
    # ── 特殊符（与 0921 一致）──
    pad_idx, sos_idx, eos_idx, unk_idx = 0, 1, 2, 3
    # ── 训练 ──
    batch_size = 64
    lr = 1e-3


print("device:", DEVICE)
print("d_k = d_v =", Cfg.d_k)

device: cuda
d_k = d_v = 32


## M1 · 两种 mask

Transformer 能「整句并行训练」，全靠 mask 挡住不该看的位置：

- **padding mask**：挡住 PAD 列。输入 `seq` 形状 `(b, l)`，输出 `(b, 1, l)`
- **因果 mask**：位置 i 只能看 0..i，不能看未来。输入长度 `l`，输出 `(1, l, l)`（下三角为 True）

自测：
1. `pad_mask` 对 `[[5, 6, 0], [7, 0, 0]]`（pad_idx = 0）应得 `[[True, True, False], [True, False, False]]`，形状 `(2, 1, 3)`
2. `causal_mask(3)` 形状 `(1, 3, 3)`，第一行 `[True, False, False]`，最后一行为全 True

想一想：`(b, 1, l)` 和 `(1, l, l)` 分别是怎么 broadcast 到注意力打分矩阵 `(b, h, lq, lk)` 上的？（M4 自测会用到）

In [3]:
def pad_mask(seq, pad_idx=Cfg.pad_idx):
    """(b, l) 词 id 序列 → (b, 1, l)；True = 真实词（可见）。"""
    return (seq != pad_idx).unsqueeze(-2) 

def causal_mask(l, device = None):
    """长度 l → (1, l, l)；下三角为 True（位置 i 只能看自己及之前）。"""
    return torch.ones(1, l, l,
                      dtype=torch.bool,
                      device=device,).tril()

# ── 自测（写好后取消注释）──
m = pad_mask(torch.tensor([[5, 6, 0], [7, 0, 0]]))
assert m.shape == (2, 1, 3)
assert m[0, 0].tolist() == [True, True, False]
assert m[1, 0].tolist() == [True, False, False]
c = causal_mask(3)
assert c.shape == (1, 3, 3)
assert c[0].tolist() == [[True, False, False],
                         [True, True, False],
                         [True, True, True]]
print("M1 OK")
print(m)
print(c)

M1 OK
tensor([[[ True,  True, False]],

        [[ True, False, False]]])
tensor([[[ True, False, False],
         [ True,  True, False],
         [ True,  True,  True]]])


## M2 · 正弦位置编码

自注意力对顺序「无感」（打乱词序结果不变），顺序信息靠位置编码注入：

$$PE_{(pos,\ 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right), \qquad PE_{(pos,\ 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

要求：
- 一次性建好 `(max_len, d_model)` 的表，用 `register_buffer` 存（不参与训练，`.to(device)` 时自动跟着搬家）
- 前向：`x + table[:l]`，形状不变

自测：
1. `(2, 5, 128)` 的零张量进，形状不变
2. 位置 0 的向量：偶维（sin）全 0、奇维（cos）全 1 —— 用来验证奇偶交错写没写对
3. 思考题：为什么用 sin/cos 三角函数，而不是可学习的位置嵌入？（提示：相对位置、外推能力）

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=Cfg.max_len):
        super().__init__()
        assert d_model % 2 == 0
        position = torch.arange(
            max_len,
            dtype=torch.float32,
        ).unsqueeze(1)
        
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32,)*
            (-math.log(10000.0)/d_model)
        )
        
        table = torch.zeros(max_len,d_model)
        table[:,0::2] = torch.sin(position*div_term)
        table[:,1::2] = torch.cos(position*div_term)
        
        self.register_buffer("table", table)
        
    def forward(self, x):
        """x: (b, l, d_model) → 同形状。"""
        seq_len = x.size(1)
        
        if seq_len > self.table.size(0):
            raise ValueError(
                f"序列长度 {seq_len} 超过位置编码上限 "
                f"{self.table.size(0)}"
            )
        return x + self.table[:seq_len].unsqueeze(0)


# ── 自测──
pe = PositionalEncoding(Cfg.d_model)
out = pe(torch.zeros(2, 5, Cfg.d_model))
assert out.shape == (2, 5, Cfg.d_model)
assert torch.allclose(out[0, 0, 0::2], torch.zeros(Cfg.d_model // 2))  # 偶维 sin(0) = 0
assert torch.allclose(out[0, 0, 1::2], torch.ones(Cfg.d_model // 2))   # 奇维 cos(0) = 1
print("M2 OK")

M2 OK


## M3 · Scaled Dot-Product Attention（单头核心）

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V$$

约定：**这一层拿到的是已经拆好头的张量**（拆头动作放在 M4）
- q `(b, h, lq, d_k)`、k `(b, h, lk, d_k)`、v `(b, h, lk, d_v)`；mask `(b, 1, lq, lk)` 或 `None`
- 返回：上下文 `(b, h, lq, d_v)` + 注意力权重 `(b, h, lq, lk)`（以后画热力图用）

三个要点：
1. 为什么除以 `√d_k`：点积的方差随 `d_k` 增大，softmax 会饱和 → 梯度变小
2. mask 的处理：把屏蔽位「填一个极大负数」再 softmax，权重自然趋近 0
3. dropout 加在注意力权重上（对照你 0921 的 dropout 位置）

自测：
1. 形状：`(2, 4, 5, 32)` 进 → 上下文 `(2, 4, 5, 32)`、权重 `(2, 4, 5, 5)` 出
2. 无 mask 时权重行和 ≈ 1
3. 遮掉 k 的最后两列后，这两列权重 ≈ 0

In [5]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        """
        q: (b, h, lq, d_k)
        k: (b, h, lk, d_k)
        v: (b, h, lk, d_v)
        mask: (b, 1, lq, lk)，True = 可见；None 表示不遮
        返回 (context (b, h, lq, d_v), attn (b, h, lq, lk))
        """
        attn = torch.matmul(q, k.transpose(2,3))
        d_k = q.size(-1)
        attn = attn / math.sqrt(d_k)
        
        if mask is not None:
            mask = mask.to(dtype=torch.bool)
            attn = attn.masked_fill(~mask, float("-inf"))
            
        weights = F.softmax(attn, dim=-1)
        dropped_weights = self.dropout(weights)
        context = torch.matmul(dropped_weights, v)
        
        return context, weights


# ── 自测（写好后取消注释）──
attn_layer = ScaledDotProductAttention()
q = torch.randn(2, 4, 5, 32); k = torch.randn(2, 4, 6, 32); v = torch.randn(2, 4, 6, 32)
ctx, w = attn_layer(q, k, v)
assert ctx.shape == (2, 4, 5, 32) and w.shape == (2, 4, 5, 6)
assert torch.allclose(w.sum(-1), torch.ones(2, 4, 5), atol=1e-5)          # 行和 ≈ 1
m = torch.ones(2, 1, 5, 6, dtype=torch.bool); m[..., -2:] = False
_, w2 = attn_layer(q, k, v, mask=m)
assert w2[..., -2:].abs().max().item() == 0.0                             # 被遮两列 ≈ 0
print("M3 OK")

M3 OK


## M4 · Multi-Head Attention

多头 = 多个子空间并行做注意力，再拼回来。张量流：

`(b, l, d_model)` → 四个线性层（W_Q / W_K / W_V / 输出投影 fc）→ `(b, l, h, d_k)` → `transpose(1, 2)` → `(b, h, l, d_k)` → M3 → `(b, h, lq, d_v)` → `transpose(1, 2)` + `view` → `(b, lq, h*d_v)` → fc → `(b, lq, d_model)`

要点：
- 「多头」不是玄学：把 d_model 拆成 h 份，各自算注意力（不同头可以学到不同的对齐关系）
- mask 传进 M3 前要补一维 —— 想想补的是哪一维（提示：M3 的打分矩阵是四维）
- 自注意力：q = k = v = 同一序列；交叉注意力：k = v = 编码器输出、q = 解码器序列

自测：
1. `(2, 7, 128)` 进 → `(2, 7, 128)` 出
2. （eval 模式下）把 v 里被 mask 的位置换成随机数，输出应完全不变 —— 这是「真遮住了」的硬核判据

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head, d_k, d_v, dropout=0.1):
        super().__init__()
        
        self.n_head = n_head
        self.d_k = d_k
        self.d_v = d_v
        
        self.w_q = nn.Linear(d_model, n_head* d_k, bias=False)
        self.w_k = nn.Linear(d_model, n_head* d_k, bias=False)
        self.w_v = nn.Linear(d_model, n_head* d_v, bias=False)
        
        self.fc = nn.Linear(n_head* d_v, d_model, bias=False)
        
        self.attention = ScaledDotProductAttention(dropout=dropout)

    def forward(self, q, k, v, mask=None):
        """
        q: (b, lq, d_model)、k: (b, lk, d_model)、v: (b, lk, d_model)
        mask: 按 M3 的语义传；补维（unsqueeze）在本函数里做
        返回 (b, lq, d_model)，顺手把 attn 权重也返回更好（画图用）
        """
        batch_size = q.size(0)
        len_q = q.size(1)
        len_k = k.size(1)
        len_v = v.size(1)
        
        if len_k != len_v:
            raise ValueError(
                f"K和V的序列长度必须相同，"
                f"现在是len_k={len_k}, len_v={len_v}"
            )
            
        q = self.w_q(q)
        k = self.w_k(k)
        v = self.w_v(v)
        
        q = q.view(batch_size, len_q, self.n_head, self.d_k).transpose(1,2)
        k = k.view(batch_size, len_k, self.n_head, self.d_k).transpose(1,2) 
        v = v.view(batch_size, len_v, self.n_head, self.d_v).transpose(1,2)
        
        if mask is not None:
            if mask.dim() == 3:
                mask = mask.unsqueeze(1)
            elif mask.dim() != 4:
                raise ValueError(
                    "mask应为3维或4维,"
                    f"实际形状是{tuple(mask.shape)}"
                )
                
        context, weights = self.attention(q, k, v, mask=mask)
        context = (context.transpose(1,2)
                   .contiguous()
                   .view(batch_size, len_q, self.n_head* self.d_v))
        
        output = self.fc(context)
        
        return output, weights

# ── 自测（写好后取消注释）──
mha = MultiHeadAttention(Cfg.d_model, Cfg.n_head, Cfg.d_k, Cfg.d_v)
out = mha(torch.randn(2, 7, 128), torch.randn(2, 7, 128), torch.randn(2, 7, 128))
out = out[0] if isinstance(out, tuple) else out
assert out.shape == (2, 7, 128)
print("M4 OK")

M4 OK


## M5 + M6 · FFN 与两种 Layer（残差 + LayerNorm）

**Position-wise FFN**：两层 Linear（`d_model → d_ff → d_model`）+ ReLU + dropout，每个位置独立算。

**EncoderLayer**（两个子层）：
`x → 自注意力 → dropout → 加残差 → LayerNorm → FFN → dropout → 加残差 → LayerNorm`

**DecoderLayer**（三个子层）：因果自注意力 → 交叉注意力（q 来自解码器，k、v 来自编码器输出）→ FFN

对照你 0921 的 Decoder 理解：那边「循环里查 hidden 的注意力」，在这里变成了「一次算完的注意力块」。

自测：
1. 三个模块出入形状不变
2. 交叉注意力的 src mask 是 `(b, 1, ls)`，进注意力前会变成什么样？（对照 M4 的补维）
3. 可选：打印每个子层输出的 `std`，观察残差 + LayerNorm 对数值尺度的作用

In [7]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """(b, l, d_model) → 同形状。"""
        hidden = self.w1(x)
        hidden = F.relu(hidden)
        hidden = self.dropout(hidden)
        output = self.w2(hidden)
        
        return output


class EncoderLayer(nn.Module):
    def __init__(self, d_model, d_ff, n_head, d_k, d_v, dropout):
        super().__init__()
        
        self.self_attn = MultiHeadAttention(d_model, n_head, d_k, d_v, dropout,)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout,)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        

    def forward(self, x, src_mask):
        """x: (b, l, d_model) → 同形状；src_mask: (b, 1, l)。"""
        attn_out, _ = self.self_attn(x, x, x, mask=src_mask,)
        x = self.norm1(x + self.dropout1(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))
        
        return x 


class DecoderLayer(nn.Module):
    def __init__(self, d_model, d_ff, n_head, d_k, d_v, dropout):
        super().__init__()
        
        self.self_attn = MultiHeadAttention(d_model, n_head, d_k, d_v, dropout,)
        self.cross_attn = MultiHeadAttention(d_model, n_head, d_k, d_v, dropout,)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout,)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_out, trg_mask, src_mask, return_attn=False):
        """x: (b, lt, d_model)、enc_out: (b, ls, d_model) → 同 x 形状。"""
        self_out, self_weights = self.self_attn(x, x, x, mask=trg_mask,)
        x = self.norm1(x + self.dropout1(self_out))
        
        cross_out, cross_weights = self.cross_attn(x, enc_out, enc_out, mask=src_mask,)
        x = self.norm2(x + self.dropout2(cross_out))
        
        ffn_out = self.ffn(x)
        x = self.norm3(x + self.dropout3(ffn_out))

        if return_attn:
            return x, self_weights, cross_weights

        return x


# ── 自测（写好后取消注释）──
el = EncoderLayer(Cfg.d_model, Cfg.d_ff, Cfg.n_head, Cfg.d_k, Cfg.d_v, Cfg.dropout)
assert el(torch.randn(2, 9, 128), torch.ones(2, 1, 9, dtype=torch.bool)).shape == (2, 9, 128)
dl = DecoderLayer(Cfg.d_model, Cfg.d_ff, Cfg.n_head, Cfg.d_k, Cfg.d_v, Cfg.dropout)
out = dl(torch.randn(2, 6, 128), torch.randn(2, 9, 128), causal_mask(6), torch.ones(2, 1, 9, dtype=torch.bool))
assert out.shape == (2, 6, 128)
print("M5/M6 OK")

M5/M6 OK


## M7 · 拼装 Encoder / Decoder / Transformer

- **Encoder**：Embedding（`padding_idx=0`）→ 乘 `√d_model`（论文做法）→ 加位置编码 → dropout → `n_layers` 个 EncoderLayer
- **Decoder**：同样构造 + DecoderLayer 堆叠
- **Transformer.forward(src, trg)**：
  1. `src_mask = pad_mask(src)`
  2. `trg_mask = pad_mask(trg)` 与 `causal_mask(trg_len)` 相与（broadcast 到一起，形状 `(b, l, l)`）
  3. `enc_out = encoder(src, src_mask)`；`dec_out = decoder(trg, enc_out, trg_mask, src_mask)`
  4. `logits = 输出线性(dec_out)` → `(b, l, 词表)`
- 训练时右移错位（和你 0921 一样）：输入 `trg[:, :-1]`（SOS 开头），监督 `trg[:, 1:]` —— 区别只是这里「一次前向算整句」，没有时间循环

自测：
1. 小词表（如 100 / 120）+ 随机 id：`src (2, 9)`、`trg (2, 6)` → logits `(2, 6, 120)`
2. 打印参数量记下来（后面调 `n_layers`、`d_model` 时对比）

In [8]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_head, d_k, d_v, d_ff, dropout, max_len):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=Cfg.pad_idx,)
        self.position = PositionalEncoding(d_model, max_len,)
        self.dropout = nn.Dropout(dropout)
        
        self.layers = nn.ModuleList([
            EncoderLayer(d_model,d_ff,n_head,d_k, d_v, dropout,)
            for _ in range(n_layers)
        ])

    def forward(self, src, src_mask):
        """src: (b, ls) 词 id → (b, ls, d_model)。"""
        x = self.embedding(src)
        x = x * math.sqrt(self.d_model)

        x = self.position(x)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, src_mask)
        return x


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_head, d_k, d_v, d_ff, dropout, max_len):
        super().__init__()
        self.d_model = d_model

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=Cfg.pad_idx,)

        self.position = PositionalEncoding(d_model, max_len,)
        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            DecoderLayer(d_model,d_ff, n_head, d_k, d_v, dropout,)
            for _ in range(n_layers)
        ])

    def forward(self, trg, enc_out, trg_mask, src_mask, return_attn=False):
        """trg: (b, lt) 词 id → (b, lt, d_model)。"""
        x = self.embedding(trg)
        x = x * math.sqrt(self.d_model)

        x = self.position(x)
        x = self.dropout(x)

        last_self_weights = None
        last_cross_weights = None

        for layer_index, layer in enumerate(self.layers):
            is_last_layer = (
                layer_index == len(self.layers) - 1
            )

            if return_attn and is_last_layer:
                (x, last_self_weights, last_cross_weights,) = layer(
                    x, enc_out, trg_mask, src_mask, return_attn=True,)
            else:
                x = layer(x, enc_out, trg_mask, src_mask,)

        if return_attn:
            return (x, last_self_weights, last_cross_weights,)

        return x


class Transformer(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, max_len):
        super().__init__()
        self.encoder = Encoder(
            vocab_size=src_vocab_size,
            d_model=Cfg.d_model,
            n_layers=Cfg.n_layers,
            n_head=Cfg.n_head,
            d_k=Cfg.d_k,
            d_v=Cfg.d_v,
            d_ff=Cfg.d_ff,
            dropout=Cfg.dropout,
            max_len=max_len,
        )

        self.decoder = Decoder(
            vocab_size=trg_vocab_size,
            d_model=Cfg.d_model,
            n_layers=Cfg.n_layers,
            n_head=Cfg.n_head,
            d_k=Cfg.d_k,
            d_v=Cfg.d_v,
            d_ff=Cfg.d_ff,
            dropout=Cfg.dropout,
            max_len=max_len,
        )

        self.output_projection = nn.Linear(
            Cfg.d_model,
            trg_vocab_size,
        )

    def forward(self, src, trg, return_attn=False):
        """返回 (b, l, trg_vocab) 的 logits。"""
        src_mask = pad_mask(src,Cfg.pad_idx,)
        trg_pad_mask = pad_mask(trg,Cfg.pad_idx,)
        trg_causal_mask = causal_mask(trg.size(1),device=trg.device,)
        trg_mask = (trg_pad_mask & trg_causal_mask)

        enc_out = self.encoder(src, src_mask,)

        if return_attn:
            (dec_out,self_weights,cross_weights,
            ) = self.decoder(trg,enc_out,trg_mask,src_mask,return_attn=True,)
        else:
            dec_out = self.decoder(
                trg,enc_out,trg_mask,src_mask,
            )
    
        logits = self.output_projection(dec_out)

        if return_attn:
            return (logits, self_weights, cross_weights,)
        return logits


# ── 自测（写好后取消注释）──
model = Transformer(100, 120, Cfg.max_len)
logits = model(torch.randint(4, 100, (2, 9)), torch.randint(4, 120, (2, 6)))
assert logits.shape == (2, 6, 120)
print("M7 OK，参数量:", sum(p.numel() for p in model.parameters()))

M7 OK，参数量: 703096


## M8 · 整体自测（接数据前最后一道防线）

两个尖锐测试，任何一个不通过都别往下走：

**① 因果性**
- 只改 `trg` 的**最后一个**词 → logits 在「最后一个词之前」的所有位置必须**完全不变**
- 改中间某个词 → 它之前的位置不变、之后的位置可变
- 原理：因果 mask 是下三角；如果「前面能影响后面」，多半是 mask 方向或 `tril / triu` 搞反了

**② 过拟合一个小 batch**
- 固定 1 个 batch，关 dropout，Adam 猛训 100~300 步，loss 应压到很近 0
- 压不下去的常见锅：右移错位没对齐、mask 接错、logits 与 gold 的长度或顺序不匹配
- 过拟合通过后，再开 dropout、接全量数据训练

In [ ]:
def check_causality(model, vocab_size=60):
    """
    检查Decoder是否严格遵守因果顺序。

    改变目标序列某个位置后：
    - 它之前的logits必须不变；
    - 它自己及之后的logits可以变化。
    """

    # 记住调用前的状态，测试后恢复
    was_training = model.training
    model.eval()

    # 保证测试张量与模型位于同一设备
    device = next(model.parameters()).device

    src = torch.randint(
        4,
        vocab_size,
        (2, 9),
        device=device,
    )

    trg = torch.randint(
        4,
        vocab_size,
        (2, 6),
        device=device,
    )

    with torch.no_grad():
        # 原始输出
        logits1 = model(src, trg)

        # ── 测试1：只改最后一个token ──
        trg2 = trg.clone()

        # 保证一定换成不同ID，同时仍然处于[4, vocab_size)范围
        trg2[:, -1] = (
            (trg2[:, -1] - 4 + 1)
            % (vocab_size - 4)
            + 4
        )

        logits2 = model(src, trg2)

        # 最后位置之前的输出必须不变
        last_prefix_diff = (
            logits1[:, :-1]
            - logits2[:, :-1]
        ).abs().max().item()

        # 最后一个位置应当受到影响
        last_position_diff = (
            logits1[:, -1]
            - logits2[:, -1]
        ).abs().max().item()

        assert torch.allclose(
            logits1[:, :-1],
            logits2[:, :-1],
            atol=1e-6,
            rtol=1e-5,
        ), (
            "因果性失败：改变最后一个token后，"
            "之前位置的logits发生了变化"
        )

        assert not torch.allclose(
            logits1[:, -1],
            logits2[:, -1],
        ), (
            "改变最后一个token后，最后位置也完全没变化，"
            "需要检查输入或测试逻辑"
        )

        # ── 测试2：改变中间位置 ──
        middle_position = trg.size(1) - 3

        trg3 = trg.clone()

        trg3[:, middle_position] = (
            (trg3[:, middle_position] - 4 + 1)
            % (vocab_size - 4)
            + 4
        )

        logits3 = model(src, trg3)

        # 被改位置之前必须不变
        middle_prefix_diff = (
            logits1[:, :middle_position]
            - logits3[:, :middle_position]
        ).abs().max().item()

        # 被改位置及之后应当受到影响
        middle_suffix_diff = (
            logits1[:, middle_position:]
            - logits3[:, middle_position:]
        ).abs().max().item()

        assert torch.allclose(
            logits1[:, :middle_position],
            logits3[:, :middle_position],
            atol=1e-6,
            rtol=1e-5,
        ), (
            "因果性失败：改变中间token后，"
            "它之前位置的logits发生了变化"
        )

        assert not torch.allclose(
            logits1[:, middle_position:],
            logits3[:, middle_position:],
        ), (
            "改变中间token后，它自己及之后的输出完全没变化"
        )

    # 恢复测试前的训练/评估状态
    model.train(was_training)

    print("最后token测试：")
    print("  前缀最大差异 =", last_prefix_diff)
    print("  最后位置差异 =", last_position_diff)

    print("中间token测试：")
    print("  前缀最大差异 =", middle_prefix_diff)
    print("  后缀最大差异 =", middle_suffix_diff)

    print("M8-① 因果性 OK")


# ── ② 过拟合小 batch（写好后取消注释）──
# model = Transformer(100, 100, Cfg.max_len)
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# x = torch.randint(4, 100, (8, 9))     # 假装是一个 batch 的 src
# y = torch.randint(4, 100, (8, 8))     # trg（接真实数据后换成真句子）
# for step in range(200):
#     optimizer.zero_grad()
#     logits = model(x, y[:, :-1])
#     loss = F.cross_entropy(logits.reshape(-1, 100), y[:, 1:].reshape(-1))
#     loss.backward()
#     optimizer.step()
#     if step % 50 == 0:
#         print(step, round(loss.item(), 4))   # 期望一路降到很小

## M9 / M10 · 接数据 → 训练 → 推理 → BLEU

数据不用重造：`0921/data.py` 已有完整链路（cmn.txt 读取 / jieba + NLTK 分词 / 词表 json / TranslationDataset / DataLoader），且词表是按 SEED=10 的训练划分建的 —— 直接复用才能和 0921 的 BLEU 0.1389 公平对照。

接法（见下一个代码格）：
```python
sys.path.insert(0, str(WEEK19 / "0921"))
import data as data_0921
train_loader, val_loader, train_pairs, val_pairs, zh_vocab, en_vocab = data_0921.create_dataloaders()
```
注意：collate 会多返回一个 `src_lengths`（Transformer 用不上，接住就行）。

训练建议（先抄 0921 配方，再做对比实验）：
- 优化器：先 Adam lr=1e-3 + ReduceLROnPlateau（与 0921 一致）；有余力再换成论文的 Noam warmup 调度对照
- label smoothing 0.1（论文）最后再加
- 监控：每轮 train / val loss 与 BLEU；权重存到自己新建的目录（如 `week19/weights_transformer/`）
- 推理：先实现 greedy（照 0921 `utils.py` 的写法改：把「逐步喂 hidden」换成「每步整句前向，取 logits 的最后一列」），beam search 作为加餐

跑完之后，回到最下面「结果记录」把数字填上。

In [ ]:
# ── M9 · 数据接入 ──
# 复用 0921 的链路：
#   sys.path.insert(0, str(WEEK19 / "0921"))
#   import data as data_0921
#   train_loader, val_loader, train_pairs, val_pairs, zh_vocab, en_vocab = data_0921.create_dataloaders()
#
# 检查清单：
#   - src：中文 + 尾部 EOS；trg：SOS + 英文 + EOS（见 data.py 的 TranslationDataset）
#   - 取一个 batch 看形状：src (b, 最长句)、trg (b, 最长句+1)、第三项 src_lengths
#   - 记下 len(zh_vocab) / len(en_vocab)，建 Transformer 时要用

import importlib
import random
import numpy as np

DATA_DIR = WEEK19 / "0921"

if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))

data_0921 = importlib.import_module("data")

# 防止意外导入了其他目录下同名的data.py
actual_data_file = Path(data_0921.__file__).resolve()
expected_data_file = (DATA_DIR / "data.py").resolve()

assert actual_data_file == expected_data_file, (
    f"导入了错误的data.py：{actual_data_file}\n"
    f"期望的是：{expected_data_file}"
)

(
    train_loader,
    val_loader,
    train_pairs,
    val_pairs,
    zh_vocab,
    en_vocab,
) = data_0921.create_dataloaders()

print("训练句对：", len(train_pairs))
print("验证句对：", len(val_pairs))
print("中文词表：", len(zh_vocab))
print("英文词表：", len(en_vocab))
print("PAD/SOS/EOS/UNK：",
      zh_vocab.PAD_INDEX,
      zh_vocab.SOS_INDEX,
      zh_vocab.EOS_INDEX,
      zh_vocab.UNK_INDEX)

In [11]:
# ── M10 · 训练循环骨架（按自己习惯补全）──
# model = Transformer(len(zh_vocab), len(en_vocab), Cfg.max_len).to(DEVICE)
# optimizer = torch.optim.Adam(model.parameters(), lr=Cfg.lr)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)
#
# for epoch in range(Cfg.num_epochs):
#     model.train()
#     for src, trg, _ in train_loader:
#         src, trg = src.to(DEVICE), trg.to(DEVICE)
#         dec_in, gold = trg[:, :-1], trg[:, 1:]              # 右移错位
#         logits = model(src, dec_in)
#         loss = F.cross_entropy(
#             logits.reshape(-1, len(en_vocab)),
#             gold.reshape(-1),
#             ignore_index=Cfg.pad_idx,                        # PAD 不产生梯度
#         )
#         optimizer.zero_grad(); loss.backward()
#         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#         optimizer.step()
#     # 验证：同构、但 model.eval() + torch.no_grad()；记录 val loss → scheduler.step(val_loss)
#     # 存 best 到 week19/weights_transformer/transformer.pt；日志写 csv

## 结果记录（训练后补）

- [ ] best val loss：____
- [ ] BLEU-4（与 0921 同口径）：____（0921 参照值 0.1389）
- [ ] 交叉注意力热力图（取最后一层）：____
- [ ] 观察与坑：____（与 jadore 版实现的差异、与 RNN 版的对比）